# Grounded Reasoning with Falcon Perception + Gemma4

Pixel-precise visual reasoning on Apple Silicon — no API keys, no cloud.

**How it works:**
1. An orchestrator VLM (Gemma4) reads your query and calls `ground_expression`.
2. **Falcon Perception** segments matching objects and returns masks + spatial metadata.
3. The VLM reasons over the Set-of-Marks image and the JSON metadata to produce a grounded answer.
4. Each example is run alongside a plain-VLM **baseline** (no tools) so you can see the difference.

```
User query + Image → Orchestrator VLM → ground_expression / compute_relations → answer
                                             ↑
                                    Falcon Perception
```

## Setup

In [ ]:
!pip install -U mlx-vlm

In [ ]:
import io
import sys
import urllib.request
from pathlib import Path

import numpy as np
from IPython.display import display
from PIL import Image

# Ensure the local mlx-vlm repo takes priority over any installed package
_HERE = Path(".").resolve()
_REPO = _HERE.parent  # mlx-vlm/
for _p in [str(_HERE), str(_REPO)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from mlx_vlm import load
from agent import GroundedReasoningResult, LocalVLMClient, run_agent, run_baseline

print("Imports OK.")

## 1 — Load Falcon Perception

In [ ]:
fp_model, fp_processor = load("tiiuae/Falcon-Perception")
print("Falcon Perception ready.")

In [ ]:
VLM_MODEL = "mlx-community/gemma-4-26b-a4b-it-8bit"  # change to suit your Mac

vlm_client = LocalVLMClient(VLM_MODEL, max_tokens=4096, temperature=0.0)
print(f"Orchestrator VLM ready ({VLM_MODEL}).")

## 3 — Helper utilities

In [ ]:
def load_image(source):
    """Load a PIL image from a local path or HTTP(S) URL."""
    if source.startswith("http"):
        with urllib.request.urlopen(source) as resp:
            return Image.open(io.BytesIO(resp.read())).convert("RGB")
    return Image.open(source).convert("RGB")


def show_result(result, title=""):
    """Print agent answer and display the supporting mask image."""
    if title:
        print(f"\n{'=' * 60}")
        print(f"  {title}")
        print(f"{'=' * 60}")
    print(f"  Answer       : {result.answer}")
    print(f"  Support masks: {result.supporting_mask_ids}")
    print(f"  FP calls     : {result.n_fp_calls}")
    print(f"  VLM calls    : {result.n_vlm_calls}")
    if result.final_image is not None:
        display(result.final_image)


def show_comparison(result, baseline_answer, query):
    """Print a side-by-side comparison of agent vs plain-VLM baseline."""
    sep = "-" * 58
    print(f"\n{'=' * 60}")
    print(f"  Query: {query}")
    print(f"{'=' * 60}")
    print(f"\n  BASELINE  (VLM only, no Falcon Perception)")
    print(f"  {sep}")
    for line in (baseline_answer or "(no response)").splitlines():
        print(f"  {line}")
    print(f"\n  GROUNDED  (VLM + Falcon Perception)")
    print(f"  {sep}")
    for line in result.answer.splitlines():
        print(f"  {line}")
    print(f"  Support masks : {result.supporting_mask_ids}")
    print(f"  FP calls      : {result.n_fp_calls}  |  VLM calls: {result.n_vlm_calls}")
    if result.final_image is not None:
        print()
        display(result.final_image)

---

## Example 1 — Spatial reasoning: *Which duck is flying the highest?*

The agent grounds all ducks, reads `centroid_norm.y` from the metadata
(smaller `y` = higher in frame), and identifies the correct one.
The baseline VLM has to judge height purely from raw pixels.

In [ ]:
image_1 = load_image(
    "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/ducks.jpg"
)
display(image_1)

In [ ]:
QUERY_1 = "Which duck is flying the highest?"

result_1 = run_agent(image_1, QUERY_1, fp_model, fp_processor, vlm_client, verbose=True)
baseline_1 = run_baseline(image_1, QUERY_1, vlm_client)
show_comparison(result_1, baseline_1, QUERY_1)

---

## Example 2 — Proximity ranking: *Which bird is closest to the camera?*

Proximity is estimated from `area_fraction` and vertical centroid position.
The agent grounds all birds and uses `compute_relations` to rank them
numerically; the baseline has to judge visually.

In [ ]:
image_2 = load_image(
    "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/seagull.jpg"
)
display(image_2)

In [ ]:
QUERY_2 = "Which bird is closest to the camera?"

result_2 = run_agent(image_2, QUERY_2, fp_model, fp_processor, vlm_client, verbose=True)
baseline_2 = run_baseline(image_2, QUERY_2, vlm_client)
show_comparison(result_2, baseline_2, QUERY_2)

---

## Example 3 — Retail shelf analysis: *Which brand has more shelf space?*

**Why grounded reasoning matters here:**
A plain VLM gives qualitative guesses — *"Coca-Cola seems to have more products."*
That is useless for a brand manager who needs numbers.

Falcon Perception gives:
- **`area_fraction` per mask** → exact share of shelf covered by each brand
- **`centroid_norm.y`** → which brand sits at eye level (the most valuable shelf position)
- **`compare_slot_positions`** → a direct side-by-side comparison of both groups

The agent grounds each brand into its own named slot, sums the area fractions,
and compares vertical positions — no visual guessing required.

In [ ]:
image_3 = load_image(
    "https://globalenterprises.online/wp-content/uploads/2024/06/pexels-photo-20155362.jpeg"
)
display(image_3)

In [ ]:
QUERY_3 = "Which brand has more shelf space, Coca-Cola or Pepsi?"

result_3 = run_agent(image_3, QUERY_3, fp_model, fp_processor, vlm_client, verbose=True)
baseline_3 = run_baseline(image_3, QUERY_3, vlm_client)
show_comparison(result_3, baseline_3, QUERY_3)

In [ ]:
image_4 = load_image(
    "https://content.api.news/v3/images/bin/11d347c590ae148c7e9b5182cf79d226?width=768"
)
display(image_4)

In [ ]:
QUERY_4 = "Is the blue player in an offside position?"

result_4 = run_agent(image_4, QUERY_4, fp_model, fp_processor, vlm_client, verbose=True)
baseline_4 = run_baseline(image_4, QUERY_4, vlm_client)
show_comparison(result_4, baseline_4, QUERY_4)

In [ ]:
image_5 = load_image(
    "https://i0.wp.com/bestsellingcarsblog.com/wp-content/uploads/2025/12/Screenshot-2025-12-23-at-7.52.32%E2%80%AFpm.jpg?w=600&ssl=1"
)
display(image_5)

In [ ]:
QUERY_5 = "Is the a french branded car in the image?"

result_5 = run_agent(image_5, QUERY_5, fp_model, fp_processor, vlm_client, verbose=True)
baseline_5 = run_baseline(image_5, QUERY_5, vlm_client)
show_comparison(result_5, baseline_5, QUERY_5)

---

## Export — Save all run results to disk

`save_run_results()` persists each example as a self-contained folder:

```
results/
  images/
    duck_original.png
    duck_final.png
    duck_step1_som.png   ← SoM per ground_expression call
    ...
  duck.json             ← query · full agent trace · baseline answer
  shelf.json
  ...
```

The JSON holds every `<think>` block, every tool call & result, the final
answer, and relative paths to all images — ready for logging, evaluation,
or sharing.

In [ ]:
from agent import save_run_results

# Map (run_name, result, baseline, query, original_image) for every example
# that has been executed above.  Skip any that haven't been run yet.
examples = [
    ("duck",        result_1,  baseline_1,  QUERY_1,  image_1),
    ("bird",        result_2,  baseline_2,  QUERY_2,  image_2),
    ("shelf",       result_3,  baseline_3,  QUERY_3,  image_3),
    ("offside",     result_4,  baseline_4,  QUERY_4,  image_4),
    ("car",         result_5,  baseline_5,  QUERY_5,  image_5),
]

OUTPUT_DIR = "./results"

for run_name, result, baseline, query, img in examples:
    try:
        path = save_run_results(
            result=result,
            baseline_answer=baseline,
            query=query,
            original_image=img,
            output_dir=OUTPUT_DIR,
            run_name=run_name,
        )
        print(f"✓  {run_name:12s} → {path}")
    except Exception as exc:
        print(f"✗  {run_name:12s} — skipped ({exc})")